In [4]:
import pandas as pd

df = pd.read_csv('Transmex_limpio.csv')

C:\Users\Emil\AppData\Local\Temp\ipykernel_61068\3934443332.py:3: DtypeWarning: Columns (3,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Transmex_student_extract.csv')


In [5]:
df.head()

,EMPTY_MILES,LOADED_MILES,TOTAL_MILES,ORDER_NUMBER,DISPATCH_NUMBER,ORDER_STATUS,NORTHBOUND_SOUTHBOUND_TYPE,LINE_OF_BUSINESS,DISPATCH_TERMINAL_LINE_OF_BUSINESS,TRUCK_TERMINAL,TRUCK_DIVISION,ORIGIN_CITY,ORIGIN_STATE,DESTINATION_CITY,DESTINATION_STATE,DISPATCH_START_DATE,DISPATCH_END_DATE,TRACTOR_NUMBER,TRAILER_NUMBER,SHIPPER_CODE
0,0,1,1,2501071,7,C,NB,N/A DRAYAGE,MEXICALI N/A DRAY,MEXIBJ,BC,MEXI,BJ,CALE,CA,2023-01-25,2023-01-25,220408,NaN,M01259
1,0,759,759,2521606,4,E,NB,NaN,NaN,NL TM,001,TOLU,EM,LARE,TX,2023-03-27,2022-11-07,1958,NaN,986186
2,0,1,1,2561454,1,E,SB,N/A DRAYAGE,LAREDO N/A DRAY,NL TM,BC,LARE,TX,ESCO,NL,2023-01-24,2023-01-24,22002,AL1311,S00514
3,0,129,129,2561454,2,E,SB,FLATBED,LAREDO FLATBED,NL TM,001,LARE,TX,ESCO,NL,2023-01-25,2023-01-26,1974,AL1311,S00514
4,0,149,149,2574176,1,E,NB,DRYVAN,LAREDO DRYVAN,NL TM,LHL,GUAD,NL,LARE,TX,2023-01-02,2023-01-02,1570,165092,S00654


In [6]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1443928 entries, 0 to 1443927
Data columns (total 20 columns):
 #   Column                              Non-Null Count    Dtype 
---  ------                              --------------    ----- 
 0   EMPTY_MILES                         1443928 non-null  int64 
 1   LOADED_MILES                        1443928 non-null  int64 
 2   TOTAL_MILES                         1443928 non-null  int64 
 3   ORDER_NUMBER                        1443928 non-null  object
 4   DISPATCH_NUMBER                     1443928 non-null  int64 
 5   ORDER_STATUS                        1443928 non-null  object
 6   NORTHBOUND_SOUTHBOUND_TYPE          1442429 non-null  object
 7   LINE_OF_BUSINESS                    1430500 non-null  object
 8   DISPATCH_TERMINAL_LINE_OF_BUSINESS  1430500 non-null  object
 9   TRUCK_TERMINAL                      1427449 non-null  object
 10  TRUCK_DIVISION                      1387176 non-null  object
 11  ORIGIN_CITY             

,EMPTY_MILES,LOADED_MILES,TOTAL_MILES,DISPATCH_NUMBER
count,1.443928e+06,1.443928e+06,1.443928e+06,1.443928e+06
mean,4.379495e+01,1.630055e+02,2.068004e+02,1.607788e+00
std,1.164608e+02,3.275516e+02,3.520897e+02,8.986659e-01
min,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
25%,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00
50%,0.000000e+00,1.000000e+00,2.700000e+01,1.000000e+00
75%,1.000000e+01,1.860000e+02,2.270000e+02,2.000000e+00
max,5.823000e+03,4.891000e+03,5.823000e+03,1.900000e+01


# ETAPA 1: EXPLORACIÓN

1.- Verificación de que no hayan datos sucios

In [7]:
df['MILES_MISMATCH'] = df['TOTAL_MILES'] != (df['EMPTY_MILES'] + df['LOADED_MILES'])

mismatches = df[df['MILES_MISMATCH']]
count_mismatches = len(mismatches)

print(f"Total de registros analizados: {len(df):,}")
print(f"Registros con discrepancias detectadas: {count_mismatches:,}")

if count_mismatches > 0:
    print("\nEjemplo de registros inconsistentes (Suma de vacías + cargadas != total):")
    display(mismatches[['EMPTY_MILES', 'LOADED_MILES', 'TOTAL_MILES']].head())
else:
    print("\nTodos los registros coinciden perfectamente.")

Total de registros analizados: 1,443,928
Registros con discrepancias detectadas: 0

Todos los registros coinciden perfectamente.


In [8]:
# Guardamos la cantidad inicial de registros
registros_iniciales = len(df)

# Eliminamos las filas con valores nulos
df_limpio = df.dropna()
registros_finales = len(df_limpio)
registros_eliminados = registros_iniciales - registros_finales

print(f"Registros iniciales: {registros_iniciales:,}")
print(f"Registros eliminados (con valores nulos): {registros_eliminados:,}")
print(f"Registros restantes limpios: {registros_finales:,}")

Registros iniciales: 1,443,928
Registros eliminados (con valores nulos): 199,724
Registros restantes limpios: 1,244,204


2.- Identificación de ordenes canceladas

In [9]:
# Filtramos directamente sobre el DataFrame limpio para que los índices coincidan perfectamente
canceladas = df_limpio[df_limpio['ORDER_STATUS'] == 'C']
count_canceladas = len(canceladas)

print(f"Total de registros limpios analizados: {len(df_limpio):,}")
print(f"Ordenes canceladas encontradas (sin nulos): {count_canceladas:,}")

Total de registros limpios analizados: 1,244,204
Ordenes canceladas encontradas (sin nulos): 7,131


3.- Exploración de empty ratio por Line of Business

In [10]:
df_limpio['EMPTY_RATIO'] = df_limpio['EMPTY_MILES'] / df_limpio['TOTAL_MILES']

df_limpio.groupby('LINE_OF_BUSINESS')['EMPTY_RATIO'].mean()

C:\Users\Emil\AppData\Local\Temp\ipykernel_61068\269032430.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_limpio['EMPTY_RATIO'] = df_limpio['EMPTY_MILES'] / df_limpio['TOTAL_MILES']


LINE_OF_BUSINESS
B1                                  0.429708
CHRYSLER                            0.510311
DEDICATED                           0.505874
DEDICATED MX                        0.498684
DEDICATED US                        0.482494
DRAYAGE                             0.475698
DRYVAN                              0.231856
ESCUELA                             0.105148
FLATBED                             0.356579
FORD                                0.235890
LAREDO DUMMY'S TRUCK THIRD PARTY    0.107143
MAZDA MX                            0.507463
MAZDA US                            0.498154
N/A DRAYAGE                         0.325813
OP1                                 0.246017
PATIO                               0.498074
REEFERS                             0.274040
SWFT TRUCKS                         0.000000
TEST                                0.171694
Name: EMPTY_RATIO, dtype: float64

4.- Orden secuenciado

In [11]:
df_limpio['ORDER_SEQ'] = df_limpio.groupby('ORDER_NUMBER').cumcount() + 1
df_limpio.head(30)

C:\Users\Emil\AppData\Local\Temp\ipykernel_61068\3695121927.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_limpio['ORDER_SEQ'] = df_limpio.groupby('ORDER_NUMBER').cumcount() + 1


,EMPTY_MILES,LOADED_MILES,TOTAL_MILES,ORDER_NUMBER,DISPATCH_NUMBER,ORDER_STATUS,NORTHBOUND_SOUTHBOUND_TYPE,LINE_OF_BUSINESS,DISPATCH_TERMINAL_LINE_OF_BUSINESS,TRUCK_TERMINAL,...,DESTINATION_CITY,DESTINATION_STATE,DISPATCH_START_DATE,DISPATCH_END_DATE,TRACTOR_NUMBER,TRAILER_NUMBER,SHIPPER_CODE,MILES_MISMATCH,EMPTY_RATIO,ORDER_SEQ
2,0,1,1,2561454,1,E,SB,N/A DRAYAGE,LAREDO N/A DRAY,NL TM,...,ESCO,NL,2023-01-24,2023-01-24,22002,AL1311,S00514,False,0.000000,1
3,0,129,129,2561454,2,E,SB,FLATBED,LAREDO FLATBED,NL TM,...,ESCO,NL,2023-01-25,2023-01-26,1974,AL1311,S00514,False,0.000000,2
4,0,149,149,2574176,1,E,NB,DRYVAN,LAREDO DRYVAN,NL TM,...,LARE,TX,2023-01-02,2023-01-02,1570,165092,S00654,False,0.000000,1
6,65,153,218,2587356,1,E,SB,OP1,LAREDO OP1,NL TM,...,ESCO,NL,2023-02-15,2023-02-15,202757,214310,S00536,False,0.298165,1
7,0,1,1,2587356,2,E,SB,N/A DRAYAGE,LAREDO N/A DRAY,NL TM,...,ESCO,NL,2023-02-15,2023-02-15,22028,214310,S00536,False,0.000000,2
9,493,447,940,2732911,1,E,NB,DRYVAN,LAREDO DRYVAN,NOGASO,...,CHIL,OH,2023-11-30,2023-12-02,24023,136242,S00244,False,0.524468,1
10,0,1409,1409,2732911,2,E,NB,OP1,LAREDO OP1,NL TM,...,CHIL,OH,2023-12-02,2023-12-04,212059,136242,S00244,False,0.000000,2
11,101,447,548,2732912,1,E,NB,DRYVAN,LAREDO DRYVAN,TEPOEM,...,LARE,TX,2023-11-30,2023-12-01,1734,234192,S00244,False,0.184307,1
13,0,1827,1827,2732913,1,E,NB,OP1,LAREDO OP1,NL TM,...,ALLE,PA,2023-11-28,2023-12-01,212058,181973,S00050,False,0.000000,1
16,0,447,447,2732915,1,E,NB,DRYVAN,LAREDO DRYVAN,ESCONL,...,LARE,TX,2023-12-01,2023-12-03,1783,150512,S00244,False,0.000000,1


5.- Datos atípicos: Mazda MX

In [12]:
# Filtrado correcto de registros para la línea de negocio 'MAZDA MX'
df_chry = df_limpio[df_limpio['LINE_OF_BUSINESS'] == 'CHRYSLER']
display(df_chry.head(30))

,EMPTY_MILES,LOADED_MILES,TOTAL_MILES,ORDER_NUMBER,DISPATCH_NUMBER,ORDER_STATUS,NORTHBOUND_SOUTHBOUND_TYPE,LINE_OF_BUSINESS,DISPATCH_TERMINAL_LINE_OF_BUSINESS,TRUCK_TERMINAL,...,DESTINATION_CITY,DESTINATION_STATE,DISPATCH_START_DATE,DISPATCH_END_DATE,TRACTOR_NUMBER,TRAILER_NUMBER,SHIPPER_CODE,MILES_MISMATCH,EMPTY_RATIO,ORDER_SEQ
843,0,211,211,2733423,1,E,NB,CHRYSLER,LAREDO CHRYSLER,NL TM,...,AUHI,MI,2023-11-29,2023-11-29,24016,TM2121,M01328,False,0.0,1
2778,0,211,211,2735064,1,E,NB,CHRYSLER,LAREDO CHRYSLER,NL TM,...,CC,TX,2023-12-02,2023-12-02,24078,TM2126,M01328,False,0.0,1
3702,0,211,211,2587571,1,E,NB,CHRYSLER,LAREDO CHRYSLER,NL TM,...,LARE,TX,2023-01-30,2023-01-30,1992,TM2124,S00189,False,0.0,1
10794,0,211,211,2588054,1,E,NB,CHRYSLER,LAREDO CHRYSLER,TIJUBJ,...,LARE,TX,2023-01-31,2023-01-31,1801,S2002,S00189,False,0.0,1
10948,0,211,211,2740921,1,E,NB,CHRYSLER,LAREDO CHRYSLER,NL TM,...,AUHI,MI,2023-12-14,2023-12-14,24091,TM2100,M01328,False,0.0,1
12428,0,211,211,2588149,1,E,NB,CHRYSLER,LAREDO CHRYSLER,NL TM,...,LARE,TX,2023-01-31,2023-01-31,1948,TM1323,S00189,False,0.0,1
12870,0,211,211,2588174,1,E,NB,CHRYSLER,LAREDO CHRYSLER,NL TM,...,LARE,TX,2023-01-31,2023-02-01,1949,S19049,S00189,False,0.0,1
12942,0,211,211,2588178,1,E,NB,CHRYSLER,LAREDO CHRYSLER,NL TM,...,LARE,TX,2023-01-31,2023-02-01,1988,TM2111,S00189,False,0.0,1
14675,0,211,211,2588265,1,E,NB,CHRYSLER,LAREDO CHRYSLER,NL TM,...,CHIC,IL,2023-01-31,2023-02-01,1951,SD1613,M01328,False,0.0,1
17521,0,211,211,2745896,1,E,NB,CHRYSLER,LAREDO CHRYSLER,NL TM,...,BELL,WA,2023-12-22,2023-12-22,2022,TM2111,M01328,False,0.0,1


# ETAPA 2: LIMPIEZA Y PERFIL POR TERMINAL

Esta sección parte del CSV original (independiente de `df`/`df_limpio` de arriba, porque `dropna()` descartaba despachos que sí sirven para el análisis). Decisiones:

- **Grano:** despacho (`ORDER_NUMBER` + `DISPATCH_NUMBER`).
- **Base de análisis:** solo `ORDER_STATUS == 'E'` (ejecutadas). `C` (canceladas), `D` y `A` (no documentadas en la guía) se separan y se reportan.
- **Terminal:** se normalizan espacios (`NL  TM`, `NL TM` y `NLTM` son la misma). Nulos → `SIN_TERMINAL`; nulos de LOB → `SIN_LOB`.
- **Fechas:** si el fin es nulo o anterior al inicio, la duración queda en `NaN` (el despacho se conserva para millas).

In [13]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

RUTA = 'Transmex_limpio.csv'
raw = pd.read_csv(RUTA, dtype=str)

for c in ['EMPTY_MILES', 'LOADED_MILES', 'TOTAL_MILES']:
    raw[c] = pd.to_numeric(raw[c])
for c in ['DISPATCH_START_DATE', 'DISPATCH_END_DATE']:
    raw[c] = pd.to_datetime(raw[c], errors='coerce')

texto = ['ORDER_NUMBER', 'DISPATCH_NUMBER', 'ORDER_STATUS', 'NORTHBOUND_SOUTHBOUND_TYPE', 'LINE_OF_BUSINESS',
         'TRUCK_TERMINAL', 'ORIGIN_CITY', 'ORIGIN_STATE', 'DESTINATION_CITY', 'DESTINATION_STATE', 'SHIPPER_CODE']
for c in texto:
    raw[c] = raw[c].str.strip()

raw['TERMINAL'] = raw['TRUCK_TERMINAL'].str.replace(r'\s+', '', regex=True).fillna('SIN_TERMINAL')
raw['LOB'] = raw['LINE_OF_BUSINESS'].fillna('SIN_LOB')
raw['DUR_DIAS'] = (raw['DISPATCH_END_DATE'] - raw['DISPATCH_START_DATE']).dt.days
raw['FECHA_VALIDA'] = raw['DISPATCH_END_DATE'].notna() & (raw['DUR_DIAS'] >= 0)
raw.loc[~raw['FECHA_VALIDA'], 'DUR_DIAS'] = np.nan
print(f"Despachos: {len(raw):,} | Órdenes: {raw['ORDER_NUMBER'].nunique():,}")

Despachos: 1,443,928 | Órdenes: 913,241


In [14]:
# Calidad: qué se detectó y qué se hizo con ello
calidad = pd.Series({
    'Total != Empty + Loaded': int((raw['TOTAL_MILES'] != raw['EMPTY_MILES'] + raw['LOADED_MILES']).sum()),
    'Fin de despacho nulo': int(raw['DISPATCH_END_DATE'].isna().sum()),
    'Fin anterior al inicio': int((raw['DUR_DIAS'].isna() & raw['DISPATCH_END_DATE'].notna()).sum()),
    'Terminal nula': int((raw['TERMINAL'] == 'SIN_TERMINAL').sum()),
    'LOB nula': int((raw['LOB'] == 'SIN_LOB').sum()),
    'Duplicados (orden + despacho)': int(raw.duplicated(['ORDER_NUMBER', 'DISPATCH_NUMBER']).sum()),
}, name='registros')
display(calidad.to_frame())

estado = raw.groupby('ORDER_STATUS').agg(despachos=('ORDER_NUMBER', 'size'), millas=('TOTAL_MILES', 'sum'),
                                         empty=('EMPTY_MILES', 'sum'))
estado['empty_ratio'] = estado['empty'] / estado['millas']
display(estado)

,registros
Total != Empty + Loaded,0
Fin de despacho nulo,1997
Fin anterior al inicio,96
Terminal nula,16479
LOB nula,13428
Duplicados (orden + despacho),0


,despachos,millas,empty,empty_ratio
ORDER_STATUS,,,,
A,186,19931,5492,0.275551
C,7459,1445106,51300,0.035499
D,3828,1464390,216823,0.148064
E,1432455,295675495,62963146,0.212947


In [15]:
base = raw[raw['ORDER_STATUS'] == 'E'].copy()
base['MES'] = base['DISPATCH_START_DATE'].dt.to_period('M')
print(f"Base de análisis (ejecutadas): {len(base):,} despachos de {len(raw):,} ({len(base)/len(raw):.1%})")
print(f"Periodo: {base['MES'].min()} a {base['MES'].max()}")

Base de análisis (ejecutadas): 1,432,455 despachos de 1,443,928 (99.2%)
Periodo: 2023-01 a 2026-09


## Perfil por terminal

`empty_ratio = Σ empty / Σ total` (como define la guía; no es el promedio de ratios por fila).

In [16]:
def resumen(g):
    return pd.Series({
        'despachos': len(g),
        'ordenes': g['ORDER_NUMBER'].nunique(),
        'millas': g['TOTAL_MILES'].sum(),
        'millas_vacias': g['EMPTY_MILES'].sum(),
        'empty_ratio': g['EMPTY_MILES'].sum() / g['TOTAL_MILES'].sum() if g['TOTAL_MILES'].sum() else np.nan,
        'millas_x_despacho': g['TOTAL_MILES'].mean(),
        'dur_media_dias': g['DUR_DIAS'].mean(),
    })

por_terminal = (base.groupby('TERMINAL').apply(resumen, include_groups=False)
                .sort_values('millas', ascending=False))
por_terminal['pct_millas'] = por_terminal['millas'] / por_terminal['millas'].sum()
por_terminal['pct_vacias'] = por_terminal['millas_vacias'] / por_terminal['millas_vacias'].sum()
display(por_terminal.round(3))

,despachos,ordenes,millas,millas_vacias,empty_ratio,millas_x_despacho,dur_media_dias,pct_millas,pct_vacias
TERMINAL,,,,,,,,,
NLTM,631277.0,437387.0,137496776.0,27498685.0,0.200,217.807,0.573,0.465,0.437
NOGASO,314837.0,234593.0,81085710.0,21528328.0,0.266,257.548,0.606,0.274,0.342
CJCI,200059.0,137203.0,25241441.0,4599628.0,0.182,126.170,0.405,0.085,0.073
MEXIBJ,72994.0,55237.0,10448462.0,1860263.0,0.178,143.141,0.410,0.035,0.030
REYNTM,39249.0,37222.0,9655794.0,1655057.0,0.171,246.014,0.696,0.033,0.026
TIJUBJ,61124.0,46637.0,8580701.0,1734861.0,0.202,140.382,0.487,0.029,0.028
TEPOEM,27905.0,25844.0,6927400.0,973945.0,0.141,248.249,0.821,0.023,0.015
ESCONL,46560.0,43435.0,6652248.0,1333478.0,0.200,142.875,0.361,0.022,0.021
SIN_TERMINAL,16433.0,15910.0,6338366.0,1063429.0,0.168,385.710,0.966,0.021,0.017


In [17]:
fig = go.Figure()
t = por_terminal.reset_index()
fig.add_bar(x=t['TERMINAL'], y=t['pct_millas'], name='% de millas totales')
fig.add_bar(x=t['TERMINAL'], y=t['pct_vacias'], name='% de millas vacías')
fig.update_layout(barmode='group', title='Peso de cada terminal: millas totales vs. millas vacías',
                  yaxis_tickformat='.0%', xaxis_title='Terminal', template='plotly_white')
fig.show()

## Terminal × Line of Business

Se ordena por millas vacías absolutas: una terminal/LOB con ratio alto pero pocas millas pesa menos que una con ratio medio y mucho volumen.

In [18]:
por_tl = (base.groupby(['TERMINAL', 'LOB']).apply(resumen, include_groups=False)
          .sort_values('millas_vacias', ascending=False))
display(por_tl.head(20).round(3))

despachos   ordenes      millas  millas_vacias  \
TERMINAL     LOB                                                            
NLTM         OP1             79394.0   75343.0  48301110.0      7289263.0   
             DRYVAN         185126.0  168500.0  61894822.0      7250929.0   
             CHRYSLER        52459.0   52235.0  11028887.0      6164711.0   
NOGASO       DEDICATED MX    55763.0   55669.0   9380991.0      4709493.0   
             OP1             51137.0   48534.0  26015854.0      4616055.0   
             DEDICATED US    37081.0   35685.0   7507615.0      3741844.0   
             DRYVAN          56607.0   52948.0  24381437.0      3646495.0   
             DRAYAGE         34430.0   34374.0   7452038.0      3532419.0   
NLTM         FLATBED         25512.0   24237.0   7858457.0      2359366.0   
CJCI         OP1             17842.0   16722.0  11666388.0      1790316.0   
             DRYVAN          50540.0   44004.0   9713013.0      1611054.0   
REYNTM       OP1             18802.0   18196.0   9433836.0      1533536.0   
NLTM         B1              26213.0   25277.0   1584063.0      1500003.0   
MEXIBJ       OP1             15897.0   15379.0   9725388.0      1476355.0   
ESCONL       DRYVAN          40194.0   37528.0   6144098.0      1064900.0   
TIJUBJ       OP1             10895.0   10565.0   6686062.0       966536.0   
NOGASO       REEFERS          7035.0    6347.0   5579422.0       944672.0   
TEPOEM       DRYVAN          24507.0   22747.0   6476758.0       810457.0   
SIN_TERMINAL OP1              7606.0    7377.0   4208114.0       663419.0   
NLTM         MAZDA MX         2380.0    2359.0   1332662.0       649032.0   

                           empty_ratio  millas_x_despacho  dur_media_dias  
TERMINAL     LOB                                                           
NLTM         OP1                 0.151            608.372           1.452  
             DRYVAN              0.117            334.339           0.805  
             CHRYSLER            0.559            210.238           0.500  
NOGASO       DEDICATED MX        0.502            168.230           0.479  
             OP1                 0.177            508.748           1.237  
             DEDICATED US        0.498            202.465           0.174  
             DRYVAN              0.150            430.714           1.074  
             DRAYAGE             0.474            216.440           0.383  
NLTM         FLATBED             0.300            308.030           0.738  
CJCI         OP1                 0.153            653.872           1.645  
             DRYVAN              0.166            192.185           0.602  
REYNTM       OP1                 0.163            501.746           1.271  
NLTM         B1                  0.947             60.430           0.336  
MEXIBJ       OP1                 0.152            611.775           1.557  
ESCONL       DRYVAN              0.173            152.861           0.383  
TIJUBJ       OP1                 0.145            613.682           1.603  
NOGASO       REEFERS             0.169            793.095           2.154  
TEPOEM       DRYVAN              0.125            264.282           0.848  
SIN_TERMINAL OP1                 0.158            553.262           1.371  
NLTM         MAZDA MX            0.487            559.942           1.188

In [19]:
top_t = por_terminal.head(8).index
top_l = base.groupby('LOB')['TOTAL_MILES'].sum().sort_values(ascending=False).head(10).index
sub = por_tl.reset_index()
sub = sub[sub['TERMINAL'].isin(top_t) & sub['LOB'].isin(top_l)]
mat = sub.pivot(index='TERMINAL', columns='LOB', values='empty_ratio').reindex(index=top_t, columns=top_l)
fig = px.imshow(mat, text_auto='.0%', aspect='auto', color_continuous_scale='Reds',
                title='Empty ratio por terminal × LOB (8 terminales y 10 LOB con más millas)')
fig.show()

## Evolución mensual por terminal

In [20]:
mensual = (base.groupby(['MES', 'TERMINAL'])
           .agg(despachos=('ORDER_NUMBER', 'size'), millas=('TOTAL_MILES', 'sum'), vacias=('EMPTY_MILES', 'sum'))
           .reset_index())
mensual['empty_ratio'] = mensual['vacias'] / mensual['millas']
mensual['MES'] = mensual['MES'].dt.to_timestamp()
mensual = mensual[mensual['TERMINAL'].isin(top_t)]

fig = px.line(mensual, x='MES', y='despachos', color='TERMINAL', title='Despachos por mes y terminal')
fig.update_layout(template='plotly_white'); fig.show()

fig = px.line(mensual, x='MES', y='empty_ratio', color='TERMINAL', title='Empty ratio mensual por terminal')
fig.update_layout(template='plotly_white', yaxis_tickformat='.0%'); fig.show()

## Recurrencia de órdenes por LOB
Una orden recurre cuando tiene más de un despacho (`DISPATCH_NUMBER`). Se analiza por LOB: cuántas órdenes toca, cuántos despachos tiene cada una dentro de ese LOB y cuántas órdenes comparten LOB con otros.

In [ ]:
import pandas as pd
import plotly.express as px

lob_df = pd.read_csv('Transmex_limpio.csv', dtype=str)
lob_df['LOB'] = lob_df['LINE_OF_BUSINESS'].str.strip()

# Despachos por orden dentro de cada LOB
por_orden_lob = (lob_df.groupby(['LOB', 'ORDER_NUMBER']).size().rename('despachos').reset_index())
# LOBs distintos por orden
lobs_por_orden = lob_df.groupby('ORDER_NUMBER')['LOB'].nunique().rename('n_lobs')
por_orden_lob = por_orden_lob.join(lobs_por_orden, on='ORDER_NUMBER')

resumen = (por_orden_lob.groupby('LOB')
           .agg(ordenes=('ORDER_NUMBER', 'size'),
                despachos=('despachos', 'sum'),
                desp_prom_por_orden=('despachos', 'mean'),
                desp_max=('despachos', 'max'),
                pct_recurrentes=('despachos', lambda s: (s > 1).mean() * 100),
                pct_multi_lob=('n_lobs', lambda s: (s > 1).mean() * 100))
           .round(2).sort_values('ordenes', ascending=False))
resumen

### Distribución de despachos por orden (por LOB)

In [ ]:
top = resumen.head(10).index
dist = (por_orden_lob[por_orden_lob.LOB.isin(top)]
        .assign(despachos=lambda d: d.despachos.clip(upper=5).astype(str).replace('5', '5+'))
        .groupby(['LOB', 'despachos']).size().rename('ordenes').reset_index())
dist['pct'] = dist['ordenes'] / dist.groupby('LOB')['ordenes'].transform('sum') * 100
px.bar(dist, x='LOB', y='pct', color='despachos', barmode='stack',
       category_orders={'LOB': list(top), 'despachos': ['1', '2', '3', '4', '5+']},
       title='% de órdenes según número de despachos dentro del LOB (top 10 LOB)').show()

### Órdenes que cruzan LOBs
Cuántas órdenes comparten cada par de LOB (una orden con varios tramos en LOBs distintos).

In [ ]:
pares = (lob_df[['ORDER_NUMBER', 'LOB']].drop_duplicates()
         .merge(lob_df[['ORDER_NUMBER', 'LOB']].drop_duplicates(), on='ORDER_NUMBER', suffixes=('_a', '_b')))
matriz = pd.crosstab(pares.LOB_a, pares.LOB_b).loc[top, top]
px.imshow(matriz, text_auto=True, aspect='auto',
          color_continuous_scale='Blues', title='Órdenes compartidas entre LOBs (diagonal = órdenes del LOB)').show()

## Recurrencia por cliente (SHIPPER_CODE), independiente del LOB
Unidad = orden única (`ORDER_NUMBER`), fechada con su primer `DISPATCH_START_DATE`. Se mide (1) cuántas órdenes y en cuántos días/meses opera cada cliente y (2) si sus órdenes siguen un patrón periódico (autocorrelación de la serie diaria de órdenes, rezagos 2–60 días). Solo clientes con ≥30 órdenes para tener señal.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

sh = pd.read_csv('Transmex_limpio.csv', dtype=str, usecols=['ORDER_NUMBER', 'SHIPPER_CODE', 'DISPATCH_START_DATE'])
sh['CLIENTE'] = sh['SHIPPER_CODE'].str.strip()
sh = sh[sh.CLIENTE != '']
sh['fecha'] = pd.to_datetime(sh['DISPATCH_START_DATE'])
ordenes = sh.groupby('ORDER_NUMBER').agg(CLIENTE=('CLIENTE', 'first'), fecha=('fecha', 'min')).reset_index()

F0, F1 = ordenes.fecha.min(), ordenes.fecha.max()
dias_total = (F1 - F0).days + 1
MIN_ORD = 30
MAX_LAG = 60

def perfil(g):
    f = g.fecha
    serie = f.dt.normalize().value_counts().reindex(pd.date_range(f.min(), f.max()), fill_value=0)
    dias_act = np.sort(serie.index[serie > 0].values)
    gaps = np.diff(dias_act).astype('timedelta64[D]').astype(int)
    x = serie.values - serie.values.mean()
    acf = {}
    if len(x) > MAX_LAG + 1 and x.var() > 0:
        den = (x * x).sum()
        acf = {l: (x[:-l] * x[l:]).sum() / den for l in range(2, MAX_LAG + 1)}
    lag = max(acf, key=acf.get) if acf else np.nan
    dow = f.dt.dayofweek.value_counts(normalize=True)
    return pd.Series({
        'ordenes': len(g), 'dias_activos': len(dias_act),
        'meses_activos': f.dt.to_period('M').nunique(),
        'primera': f.min(), 'ultima': f.max(), 'span_dias': (f.max() - f.min()).days + 1,
        'ord_por_dia_activo': len(g) / len(dias_act),
        'gap_mediano': np.median(gaps) if len(gaps) else np.nan,
        'gap_cv': gaps.std() / gaps.mean() if len(gaps) and gaps.mean() > 0 else np.nan,
        'cobertura': len(dias_act) / ((f.max() - f.min()).days + 1),
        'dow_dominante': dow.idxmax(), 'dow_share': dow.max(),
        'lag_pico': lag, 'acf_pico': acf.get(lag, np.nan),
        'acf_7': acf.get(7, np.nan), 'acf_14': acf.get(14, np.nan), 'acf_30': acf.get(30, np.nan)})

cli = ordenes.groupby('CLIENTE').filter(lambda g: len(g) >= MIN_ORD).groupby('CLIENTE').apply(perfil, include_groups=False)
print(f'{ordenes.CLIENTE.nunique()} clientes en total; {len(cli)} con >= {MIN_ORD} órdenes '
      f'({cli.ordenes.sum() / len(ordenes):.1%} de las órdenes)')
cli.sort_values('ordenes', ascending=False).head(15).round(2)

### Clasificación de periodicidad
Se declara periódico si el pico de autocorrelación (rezagos 2–60) supera 0.25. Se etiqueta *semanal* si el pico cae en 7/14/21/28 (±1), *mensual* si cae en 28–31 y *otro periodo* en el resto. Sin periodicidad y con actividad en más del 80% de los días del span: *continuo sin ciclo* (opera todos los días de forma pareja).

In [ ]:
def clasifica(r):
    if pd.isna(r.acf_pico) or r.acf_pico < 0.25:
        return 'Continuo sin ciclo' if r.cobertura > 0.8 else 'Sin periodicidad'
    l = int(r.lag_pico)
    if l % 7 in (0, 1, 6) and l <= 28 and l >= 6:
        return 'Semanal'
    if 28 <= l <= 31:
        return 'Mensual'
    return 'Otro periodo'

cli['patron'] = cli.apply(clasifica, axis=1)
res = (cli.groupby('patron').agg(clientes=('ordenes', 'size'), ordenes=('ordenes', 'sum'),
                                 acf_pico_med=('acf_pico', 'median'), gap_mediano=('gap_mediano', 'median'))
       .assign(pct_clientes=lambda d: d.clientes / d.clientes.sum() * 100,
               pct_ordenes=lambda d: d.ordenes / d.ordenes.sum() * 100).round(2))
res

In [ ]:
px.histogram(cli.reset_index(), x='acf_pico', color='patron', nbins=40,
             title='Fuerza de la periodicidad por cliente (pico de autocorrelación, rezagos 2-60 días)').show()
px.scatter(cli.reset_index(), x='lag_pico', y='acf_pico', size='ordenes', color='patron', hover_name='CLIENTE',
           title='Periodo dominante (días) vs fuerza, por cliente').show()

### Distribución de recurrencia: órdenes por cliente y antigüedad

In [ ]:
px.histogram(cli.reset_index(), x='ordenes', log_x=True, nbins=40, title='Órdenes por cliente (clientes ≥30 órdenes)').show()
px.scatter(cli.reset_index(), x='meses_activos', y='ordenes', log_y=True, color='patron', hover_name='CLIENTE',
           title='Meses activos vs órdenes').show()

### Top 6 clientes: órdenes semanales (para inspección visual de periodicidad)

In [ ]:
top6 = cli.sort_values('ordenes', ascending=False).head(6).index
sem = (ordenes[ordenes.CLIENTE.isin(top6)]
       .groupby([pd.Grouper(key='fecha', freq='W'), 'CLIENTE']).size().rename('ordenes').reset_index())
px.line(sem, x='fecha', y='ordenes', facet_row='CLIENTE', height=900,
        title='Órdenes semanales de los 6 clientes principales').update_yaxes(matches=None).show()
dow = (ordenes[ordenes.CLIENTE.isin(top6)].assign(dia=lambda d: d.fecha.dt.day_name())
       .groupby(['CLIENTE', 'dia']).size().rename('ordenes').reset_index())
px.bar(dow, x='dia', y='ordenes', facet_col='CLIENTE', facet_col_wrap=3,
       category_orders={'dia': ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']},
       title='Órdenes por día de la semana, top 6 clientes').update_yaxes(matches=None).show()